# Synthetic Datasets: Plots of Individual Fairness Metrics

Author: Ilse Harmers \
Last modified: March 17, 2026

Certain sections of code in this notebook have been kindly provided by dr. Mina Alishahi.

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

import matplotlib
params = {'axes.titlesize':'20',
          'xtick.labelsize':'18',
          'ytick.labelsize':'18',
          'font.size':'18',
          'legend.fontsize':'medium',
          'lines.linewidth':'2.0',
          'font.weight':'normal',
          'lines.markersize':'5',
          'text.latex.preamble': r'\usepackage{amsfonts}',
          }
matplotlib.rcParams.update(params)
plt.rcParams["mathtext.fontset"] = "cm"
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier

## Initialization

In [ ]:
# Initializing the real datasets (Adult, Bank Marketing and Credit Card Default), our classifier and our functions. 
adult_train_encoded = pd.read_csv("./train-test-datasets/adult/train_encoded.csv")
adult_test_encoded = pd.read_csv("./train-test-datasets/adult/test_encoded.csv")
bank_train_encoded = pd.read_csv("./train-test-datasets/bank-marketing/train_encoded.csv")
bank_test_encoded = pd.read_csv("./train-test-datasets/bank-marketing/test_encoded.csv")
credit_train_encoded = pd.read_csv("./train-test-datasets/credit-card-default/train_encoded.csv")
credit_test_encoded = pd.read_csv("./train-test-datasets/credit-card-default/test_encoded.csv")

# Initializing our classifier.
rf_clf = RandomForestClassifier()

def encode_adult(sample):
    """This function encodes an Adult-based synthetic dataset such that all ordinal columns are scaled to a range of (0, 1)."""

    sample["education-num"] = ((sample["education-num"] - sample["education-num"].min()) / 
                               (sample["education-num"].max() - sample["education-num"].min()))

    return sample

def encode_bank(sample):
    """This function encodes a Bank-based synthetic dataset such that all ordinal columns are scaled to a range of (0, 1)."""

    sample["day"] = ((sample["day"] - sample["day"].min()) / (sample["day"].max() - sample["day"].min()))
    
    return sample
    
def encode_credit(sample):
    """This function encodes a Credit-based synthetic dataset such that all ordinal columns are scaled to a range of (0, 1)."""

    sample["SEX"] = ((sample["SEX"] - sample["SEX"].min()) / (sample["SEX"].max() - sample["SEX"].min()))
    
    for p in [0, 2, 3, 4, 5, 6]:
        sample[f"PAY_{p}"] = ((sample[f"PAY_{p}"] - sample[f"PAY_{p}"].min()) / 
                              (sample[f"PAY_{p}"].max() - sample[f"PAY_{p}"].min()))

    return sample

def compute_similarity_fairness(indices, predictions, distances, neighbors):
    """
    Compute individual fairness for a chunk of data using precomputed k-NN distances and neighbors.

    Parameters:
        indices (list): Indices of the chunk.
        predictions (array-like): Model predictions for individuals.
        distances (array-like): Distances from k-NN.
        neighbors (array-like): Indices of k-NN neighbors.

    Returns:
        float: Mean individual fairness score for the chunk.

    Code provided by: dr. Mina Alishahi
    """
    
    fairness_scores = []
    for idx in indices:
        # Iterate through the k-nearest neighbors (skip the first as it's the individual itself).
        for neighbor_idx, distance in zip(neighbors[idx][1:], distances[idx][1:]):
            if distance < 1e-6:  # Treat very small distances as non-zero.
                distance = 1e-6
            pred_diff = abs(predictions[idx] - predictions[neighbor_idx])
            fairness_scores.append(pred_diff * distance)
    return np.mean(fairness_scores) if fairness_scores else 0.0

def similarity_fairness(predictions, features, similarity_metric = "euclidean", k = 100, num_chunks = 8):
    """
    Compute individual fairness for model predictions using k-NN.

    Parameters:
        predictions (array-like): Model predictions for individuals.
        features (array-like): Feature matrix (N x D).
        similarity_metric (str): Similarity metric ('cosine' or 'euclidean').
        k (int): Number of nearest neighbors to consider.
        num_chunks (int): Number of chunks.

    Returns:
        float: Average individual fairness score (lower is better).

    Code provided by: dr. Mina Alishahi
    """
    
    n = len(features)
    if k >= n:
        raise ValueError(f"Invalid k: {k}. Must be less than the number of data points: {n}.")
    
    # Fit k-NN on the features.
    np.random.seed(42)   # setting random seed
    knn = NearestNeighbors(n_neighbors = k + 1, metric = similarity_metric).fit(features)
    distances, neighbors = knn.kneighbors(features, return_distance=True)

    # Divide indices into chunks.
    chunk_size = max(1, n // num_chunks)
    fairness_scores = [
        compute_similarity_fairness(
            range(start, min(start + chunk_size, n)), predictions, distances, neighbors
        )
        for start in range(0, n, chunk_size)
    ]

    # Gather results from all chunks and compute the overall mean.
    return np.mean(fairness_scores) if fairness_scores else 0.0


def compute_neighborhood_consistency_fairness(indices, predictions, distances, neighbors):
    """
    Compute neighborhood consistency for a chunk of data.

    Parameters:
        indices (list): Indices of the chunk.
        predictions (array-like): Model predictions for individuals.
        distances (array-like): Distances from k-NN.
        neighbors (array-like): Indices of k-NN neighbors.

    Returns:
        float: Mean neighborhood consistency score for the chunk.

    Code provided by: dr. Mina Alishahi
    """
    consistency_scores = []
    for idx in indices:
        # Calculate consistency score for the current individual.
        local_consistency = np.mean([
            abs(predictions[idx] - predictions[neighbor_idx])
            for neighbor_idx in neighbors[idx][1:]  # Skip the first neighbor (the individual itself).
        ])
        consistency_scores.append(local_consistency)

    return np.mean(consistency_scores)

def neighborhood_consistency_fairness(predictions, features, similarity_metric = "euclidean", k = 100, num_chunks = 8):
    """
    Compute neighborhood consistency metric using k-NN.

    Parameters:
        predictions (array-like): Model predictions for individuals.
        features (array-like): Feature matrix (N x D).
        similarity_metric (str): Similarity metric ('cosine' or 'euclidean').
        k (int): Number of nearest neighbors to consider.
        num_chunks (int): Number of chunks.

    Returns:
        float: Average neighborhood consistency score (lower is better).

    Code provided by: dr. Mina Alishahi
    """
        
    # Fit k-NN on the features.
    np.random.seed(42)   # setting random seed
    knn = NearestNeighbors(n_neighbors = k + 1, metric = similarity_metric).fit(features)
    distances, neighbors = knn.kneighbors(features, return_distance = True)

    # Divide indices into chunks.
    n = len(features)
    chunk_size = n // num_chunks
    consistency_scores = [
        compute_neighborhood_consistency_fairness(
            range(start, min(start + chunk_size, n)), predictions, distances, neighbors
        )
        for start in range(0, n, chunk_size)
    ]
    
    # Gather results from all chunks and compute the overall mean.
    return np.mean(consistency_scores)

def fairness_analysis(train_set, test_set, dataset):
    """This function computes the similarity fairness and neigborhood consistency fairness scores for a given train and test set.
    
    train_set [array-like]: train set 
    test_set [array-like]: test set
    dataset [string]: dataset name; can be set to "Adult", "Bank" or "Credit"
    """

    if dataset == "Adult":
        y = "income"
    elif dataset == "Bank":
        y = "y"
    elif dataset == "Credit":
        y = "DEFAULT"
    else:
        raise ValueError("Dataset not supported.")

    # Training a random forest classifier and getting its predictions.
    np.random.seed(42)   # setting random seed
    rf_model = rf_clf.fit(train_set.drop(columns = [y]), train_set[y])
    soft_ypred = rf_model.predict_proba(test_set.drop(columns = [y]))
    ypred = rf_model.predict(test_set.drop(columns = [y]))

    # Min-max scaling the ordinal columns.
    if dataset == "Adult":
        test_copy = test_set.copy()
        test_enc = encode_adult(test_copy)
    elif dataset == "Bank":
        test_copy = test_set.copy()
        test_enc = encode_bank(test_copy)
    elif dataset == "Credit":
        test_copy = test_set.copy()
        test_enc = encode_credit(test_copy)
    else:
        raise ValueError("Dataset not supported.")
        
    sf_score = similarity_fairness(soft_ypred, test_enc.drop(columns = [y]).values)

    ncf_score = neighborhood_consistency_fairness(ypred, test_enc.drop(columns = [y]).values)

    return (sf_score, ncf_score)

def results_analysis(file_path, epsi, test_set, y, dataset):
    """This function computes the similarity fairness and neigborhood consistency fairness scores for a given GAN model, list of epsilon values and test 
    set.

    If the GAN model does not have privacy constraints, set 'epsi' = [].
    
    file_path [string]: file path to synthetic datasets of a GAN model 
    epsi [list]: epsilon values
    test_set [array-like]: test set
    y [string]: column name of target attribute
    dataset [string]: name of the dataset; can be set to "Adult", "Bank" or "Credit"
    """

    idf_scores = {}   # individual fairness scores
    runs = range(1, 6)  
    samples = range(1, 4)
    c = 1
    # Computing the individual fairness scores for the synthetic datasets of a GAN model.
    for r in runs:
        for s in samples:
            if epsi:
                synthetic_dataset = pd.read_csv(file_path + f"epsi_{epsi}/run{r}/sample{s}_encoded.csv")
            else:
                synthetic_dataset = pd.read_csv(file_path + f"run{r}/sample{s}_encoded.csv")

            # Training a random forest classifier and getting its predictions.
            np.random.seed(42)   # setting random seed
            rf_model = rf_clf.fit(synthetic_dataset.drop(columns = [y]), synthetic_dataset[y])
            ypred = rf_model.predict(test_set.drop(columns = [y]))
            soft_ypred = rf_model.predict_proba(test_set.drop(columns = [y])) 

            # Min-max scaling the ordinal columns.
            if dataset == "Adult":
                test_copy = test_set.copy()
                test_enc = encode_adult(test_copy)
            elif dataset == "Bank":
                test_copy = test_set.copy()
                test_enc = encode_bank(test_copy)
            elif dataset == "Credit":
                test_copy = test_set.copy()
                test_enc = encode_credit(test_copy)
            else:
                raise ValueError("Dataset not supported.")
            
            sf_score = similarity_fairness(soft_ypred, test_enc.drop(columns = [y]).values)
            ncf_score = neighborhood_consistency_fairness(ypred, test_enc.drop(columns = [y]).values)
            idf_scores[f"sample{c}"] = (np.abs(sf_score), np.abs(ncf_score))
            
            c += 1
            
    return pd.DataFrame(idf_scores).rename(index = {0: "SF", 1: "NCF"})

def print_results(file_path, dataset):
    """Printing and saving the individual fairness scores of a GAN model.
    
    file_path [string]: file path to synthetic datasets of a GAN model
    dataset [string]: dataset name; can be set to "Adult", "Bank" or "Credit"
    """
    
    if dataset == "Adult":
        test_set = adult_test_encoded
        y = "income"
    elif dataset == "Bank":
        test_set = bank_test_encoded
        y = "y"
    elif dataset == "Credit":
        test_set = credit_test_encoded
        y = "DEFAULT"
    else:
        print("Dataset not supported.")

    # Computing the individual fairness results for each epsilon value.
    idf1_scores = results_analysis(file_path = file_path, epsi = 1, test_set = test_set, y = y, dataset = dataset)
    idf2_scores = results_analysis(file_path = file_path, epsi = 2, test_set = test_set, y = y, dataset = dataset)
    idf5_scores = results_analysis(file_path = file_path, epsi = 5, test_set = test_set, y = y, dataset = dataset)
    idf8_scores = results_analysis(file_path = file_path, epsi = 8, test_set = test_set, y = y, dataset = dataset)

    # Saving the individual fairness scores as csv files.
    idf1_scores.to_csv(file_path + "indfair-scores_epsi-1.csv")
    idf2_scores.to_csv(file_path + "indfair-scores_epsi-2.csv")
    idf5_scores.to_csv(file_path + "indfair-scores_epsi-5.csv")
    idf8_scores.to_csv(file_path + "indfair-scores_epsi-8.csv")
    
    # Printing the average individual fairness scores. 
    print("SF results")
    print(np.nanmean(idf1_scores.iloc[0]), np.nanstd(idf1_scores.iloc[0]))
    print(np.nanmean(idf2_scores.iloc[0]), np.nanstd(idf2_scores.iloc[0]))
    print(np.nanmean(idf5_scores.iloc[0]), np.nanstd(idf5_scores.iloc[0]))
    print(np.nanmean(idf8_scores.iloc[0]), np.nanstd(idf8_scores.iloc[0]))
    print()
    print("NCF results")
    print(np.nanmean(idf1_scores.iloc[1]), np.nanstd(idf1_scores.iloc[1]))
    print(np.nanmean(idf2_scores.iloc[1]), np.nanstd(idf2_scores.iloc[1]))
    print(np.nanmean(idf5_scores.iloc[1]), np.nanstd(idf5_scores.iloc[1]))
    print(np.nanmean(idf8_scores.iloc[1]), np.nanstd(idf8_scores.iloc[1]))

def ind_fair_results(model, epsi, samples, dataset):
    """Importing the individual fairness results for a GAN model when given a list of 'best' synthetic datasets. 

    If the GAN model does not have privacy constraints, set 'epsi' = [].

    model [string]: name of GAN model
    epsi [list]: epsilon values
    samples [list]: "names" of the synthetic datasets
    dataset [string]: folder name of the synthetic datasets; can be set to "adult", "bank-marketing" or "credit-card-default"
    """

    if model == "TabFair":
        ind_fairs = pd.DataFrame(pd.read_csv(f"./synthetic-datasets_{model}/{dataset}/indfair-scores.csv", index_col = 0).T[f"sample{samples[0]}"]).T
    elif model == "DP-GAN" or model == "FairDP-GAN(dis)":
        ind_fairs = pd.concat([pd.DataFrame(pd.read_csv(f"../synthetic-datasets_{model}_B=512/{dataset}/indfair-scores_epsi-{epsi[e]}.csv", 
                                                        index_col = 0)[f"sample{samples[e]}"]).T for e in range(len(epsi))], axis = 0)
    else:
        ind_fairs = pd.concat([pd.DataFrame(pd.read_csv(f"./synthetic-datasets_{model}/{dataset}/indfair-scores_epsi-{epsi[e]}.csv", 
                                                        index_col = 0)[f"sample{samples[e]}"]).T for e in range(len(epsi))], axis = 0)
    return ind_fairs

## Adult Dataset (Real)

In [ ]:
# Individual fairness analysis of Adult dataset.
adult_sf, adult_ncf = fairness_analysis(train_set = adult_train_encoded, test_set = adult_test_encoded, dataset = "Adult")
print("sf_score: ", adult_sf)  
print("ncf_score: ", adult_ncf)

adult_results = pd.DataFrame({"SF": [adult_sf], "NCF": [adult_ncf]})

In [ ]:
print(adult_sf, adult_ncf)
adult_results

## Synthetic Adult Dataset (DP-GAN)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_DP-GAN_B=512/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (*Dis* with $\lambda = 0.5$)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dis)_B=512/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (*Dis* with $\lambda = 0.75$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=0.75/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (*Dis* with $\lambda = 1.0$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=1.0/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (*Dis* with $\lambda = 1.25$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=1.25/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (*Dis* with $\lambda = 1.5$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=1.5/adult/"
print_results(file_path, dataset = "Adult")

## Bank Marketing Dataset (Real)

In [ ]:
# Individual fairness analysis of Bank Marketing dataset.
bank_sf, bank_ncf = fairness_analysis(train_set = bank_train_encoded, test_set = bank_test_encoded, dataset = "Bank")
print("sf_score: ", bank_sf)  
print("ncf_score: ", bank_ncf)

bank_results = pd.DataFrame({"SF": [bank_sf], "NCF": [bank_ncf]})

In [ ]:
print(bank_sf, bank_ncf)
bank_results

## Synthetic Bank Marketing Dataset (DP-GAN)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_DP-GAN_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 0.5$)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dis)_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 0.75$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=0.75/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 1.0$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=1.0/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 1.25$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=1.25/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 1.5$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=1.5/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Credit Card Default Dataset (Real)

In [ ]:
# Individual fairness analysis of Credit Card Default dataset.
credit_sf, credit_ncf = fairness_analysis(train_set = credit_train_encoded, test_set = credit_test_encoded, dataset = "Credit")
print("sf_score: ", credit_sf)  
print("ncf_score: ", credit_ncf)

credit_results = pd.DataFrame({"SF": [credit_sf], "NCF": [credit_ncf]})

In [ ]:
print(credit_sf, credit_ncf)
credit_results

## Synthetic Credit Card Default Dataset (DP-GAN)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_DP-GAN_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 0.5$)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dis)_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 0.75$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=0.75/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 1.0$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=1.0/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 1.25$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=1.25/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 1.5$)

In [ ]:
# Assigning file path.
file_path = "./synthetic-datasets_FairDP-GAN(dis)_l=1.5/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Results of *Best* Synthetic Datasets

In [ ]:
epsi = [1, 2, 5, 8]   # epsilon values

### Adult Dataset

In [ ]:
# See Group-Fairness-Utility_Plots.ipynb for the 'best' synthetic datasets that are reported here for the parameter 'samples'.

adult_DP = ind_fair_results(model = "DP-GAN", epsi = epsi, samples = [1, 1, 5, 10], dataset = "adult")

adult_dis = ind_fair_results(model = "FairDP-GAN(dis)", epsi = epsi, samples = [4, 7, 6, 3], dataset = "adult")
adult_075 = ind_fair_results(model = "FairDP-GAN(dis)_l=0.75", epsi = epsi, samples = [1, 13, 6, 6], dataset = "adult")
adult_100 = ind_fair_results(model = "FairDP-GAN(dis)_l=1.0", epsi = epsi, samples = [13, 6, 9, 6], dataset = "adult")
adult_125 = ind_fair_results(model = "FairDP-GAN(dis)_l=1.25", epsi = epsi, samples = [9, 9, 10, 7], dataset = "adult")
adult_150 = ind_fair_results(model = "FairDP-GAN(dis)_l=1.5", epsi = epsi, samples = [2, 10, 10, 15], dataset = "adult")

### Bank Marketing Dataset

In [ ]:
# See Group-Fairness-Utility_Plots.ipynb for the 'best' synthetic datasets that are reported here for the parameter 'samples'.

bank_DP = ind_fair_results(model = "DP-GAN", epsi = epsi, samples = [2, 5, 8, 6], dataset = "bank-marketing")

bank_dis = ind_fair_results(model = "FairDP-GAN(dis)", epsi = epsi, samples = [2, 11, 15, 3], dataset = "bank-marketing")
bank_075 = ind_fair_results(model = "FairDP-GAN(dis)_l=0.75", epsi = epsi, samples = [12, 12, 2, 14], dataset = "bank-marketing")
bank_100 = ind_fair_results(model = "FairDP-GAN(dis)_l=1.0", epsi = epsi, samples = [1, 14, 8, 10], dataset = "bank-marketing")
bank_125 = ind_fair_results(model = "FairDP-GAN(dis)_l=1.25", epsi = epsi, samples = [8, 14, 15, 8], dataset = "bank-marketing")
bank_150 = ind_fair_results(model = "FairDP-GAN(dis)_l=1.5", epsi = epsi, samples = [11, 5, 9, 2], dataset = "bank-marketing")

### Credit Card Default Dataset

In [ ]:
# See Group-Fairness-Utility_Plots.ipynb for the 'best' synthetic datasets that are reported here for the parameter 'samples'.

credit_DP = ind_fair_results(model = "DP-GAN", epsi = epsi, samples = [11, 4, 4, 6], dataset = "credit-card-default")

credit_dis = ind_fair_results(model = "FairDP-GAN(dis)", epsi = epsi, samples = [9, 13, 2, 13], dataset = "credit-card-default")
credit_075 = ind_fair_results(model = "FairDP-GAN(dis)_l=0.75", epsi = epsi, samples = [7, 14, 8, 6], dataset = "credit-card-default")
credit_100 = ind_fair_results(model = "FairDP-GAN(dis)_l=1.0", epsi = epsi, samples = [3, 7, 13, 5], dataset = "credit-card-default")
credit_125 = ind_fair_results(model = "FairDP-GAN(dis)_l=1.25", epsi = epsi, samples = [9, 5, 14, 3], dataset = "credit-card-default")
credit_150 = ind_fair_results(model = "FairDP-GAN(dis)_l=1.5", epsi = epsi, samples = [1, 8, 10, 5], dataset = "credit-card-default")

### Average of Results

In [ ]:
# Average of datasets' SF results.
avg_sf_real = [adult_results["SF"], bank_results["SF"], credit_results["SF"]]

avg_sf_DP = [[adult_DP["SF"][e], bank_DP["SF"][e], credit_DP["SF"][e]] for e in range(0, 4)]

avg_sf_dis = [[adult_dis["SF"][e], bank_dis["SF"][e], credit_dis["SF"][e]] for e in range(0, 4)]
avg_sf_075 = [[adult_075["SF"][e], bank_075["SF"][e], credit_075["SF"][e]] for e in range(0, 4)]
avg_sf_100 = [[adult_100["SF"][e], bank_100["SF"][e], credit_100["SF"][e]] for e in range(0, 4)]
avg_sf_125 = [[adult_125["SF"][e], bank_125["SF"][e], credit_125["SF"][e]] for e in range(0, 4)]
avg_sf_150 = [[adult_150["SF"][e], bank_150["SF"][e], credit_150["SF"][e]] for e in range(0, 4)]

# Average of datasets' NCF results.
avg_ncf_real = [adult_results["NCF"], bank_results["NCF"], credit_results["NCF"]]

avg_ncf_DP = [[adult_DP["NCF"][e], bank_DP["NCF"][e], credit_DP["NCF"][e]] for e in range(0, 4)]

avg_ncf_dis = [[adult_dis["NCF"][e], bank_dis["NCF"][e], credit_dis["NCF"][e]] for e in range(0, 4)]
avg_ncf_075 = [[adult_075["NCF"][e], bank_075["NCF"][e], credit_075["NCF"][e]] for e in range(0, 4)]
avg_ncf_100 = [[adult_075["NCF"][e], bank_100["NCF"][e], credit_100["NCF"][e]] for e in range(0, 4)]
avg_ncf_125 = [[adult_125["NCF"][e], bank_125["NCF"][e], credit_125["NCF"][e]] for e in range(0, 4)]
avg_ncf_150 = [[adult_150["NCF"][e], bank_150["NCF"][e], credit_150["NCF"][e]] for e in range(0, 4)]

## Plots of Results

In [ ]:
# Plotting individual fairness metrics of the 'best' synthetic datasets from all our models.
labels = ["Original", "DP-GAN", "$Dis$ ($\lambda_f = 0.5$)", "$Dis$ ($\lambda_f = 0.75$)", "$Dis$ ($\lambda_f = 1.0$)", "$Dis$ ($\lambda_f = 1.25$)",
          "$Dis$ ($\lambda_f = 1.5$)"]
colors = ["red", "purple", "darkcyan", "darkorange", "olive", "magenta", "royalblue"]
markers = ["o", "D", "P", "^", "s", "v"]
msize = 9

limits = {"SF": [(-0.005, 0.25)], "NCF": [(-0.005, 0.29)]}

fairness_clf = ["SF", "NCF"]
fairness_labels = ["ASF", "ANCF"]

for f in fairness_clf:
    # Create a figure for the aggregated plot.
    fig, axes = plt.subplots(1, 4, figsize = (20, 8.0))  # 1 x 4 grid
    for c in range(0, 4):
        if c == 0:
            dataset = "Adult"
            results_np = pd.DataFrame(adult_results[f]).set_axis(["Real"], axis = 1).set_axis([f], axis = 0)
                
            results_DP = pd.DataFrame(adult_DP[f]).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.DataFrame(adult_dis[f]).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.DataFrame(adult_075[f]).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)

            results_f100DP = pd.DataFrame(adult_100[f]).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)

            results_f125DP = pd.DataFrame(adult_125[f]).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)

            results_f150DP = pd.DataFrame(adult_150[f]).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        elif c == 1:
            dataset = "Bank"
            results_np = pd.DataFrame(bank_results[f]).set_axis(["Real"], axis = 1).set_axis([f], axis = 0)
                
            results_DP = pd.DataFrame(bank_DP[f]).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.DataFrame(bank_dis[f]).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.DataFrame(bank_075[f]).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)

            results_f100DP = pd.DataFrame(bank_100[f]).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)

            results_f125DP = pd.DataFrame(bank_125[f]).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)

            results_f150DP = pd.DataFrame(bank_150[f]).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        elif c == 2:
            dataset = "Credit"
            results_np = pd.DataFrame(credit_results[f]).set_axis(["Real"], axis = 1).set_axis([f], axis = 0)
                
            results_DP = pd.DataFrame(credit_DP[f]).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.DataFrame(credit_dis[f]).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.DataFrame(credit_075[f]).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)

            results_f100DP = pd.DataFrame(credit_100[f]).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)

            results_f125DP = pd.DataFrame(credit_125[f]).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)

            results_f150DP = pd.DataFrame(credit_150[f]).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)
            
        else:
            dataset = "Average"
            if f == "SF":
                results = pd.DataFrame({"mean": [np.mean(avg_sf_real)] + list(np.mean(avg_sf_DP, axis = 1)),
                                        "std": [np.std(avg_sf_real)] + list(np.std(avg_sf_DP, axis = 1))},
                                       index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

                results_fdis = pd.DataFrame({"mean": [np.mean(avg_sf_real)] + list(np.mean(avg_sf_dis, axis = 1)),
                                             "std": [np.std(avg_sf_real)] + list(np.std(avg_sf_dis, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f075 = pd.DataFrame({"mean": [np.mean(avg_sf_real)] + list(np.mean(avg_sf_075, axis = 1)),
                                             "std": [np.std(avg_sf_real)] + list(np.std(avg_sf_075, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f100 = pd.DataFrame({"mean": [np.mean(avg_sf_real)] + list(np.mean(avg_sf_100, axis = 1)),
                                             "std": [np.std(avg_sf_real)] + list(np.std(avg_sf_100, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f125 = pd.DataFrame({"mean": [np.mean(avg_sf_real)] + list(np.mean(avg_sf_125, axis = 1)),
                                             "std": [np.std(avg_sf_real)] + list(np.std(avg_sf_125, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f150 = pd.DataFrame({"mean": [np.mean(avg_sf_real)] + list(np.mean(avg_sf_150, axis = 1)),
                                             "std": [np.std(avg_sf_real)] + list(np.std(avg_sf_150, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
            else: 
                results = pd.DataFrame({"mean": [np.mean(avg_ncf_real)] + list(np.mean(avg_ncf_DP, axis = 1)),
                                        "std": [np.std(avg_ncf_real)] + list(np.std(avg_ncf_DP, axis = 1))},
                                       index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                
                results_fdis = pd.DataFrame({"mean": [np.mean(avg_ncf_real)] + list(np.mean(avg_ncf_dis, axis = 1)),
                                             "std": [np.std(avg_ncf_real)] + list(np.std(avg_ncf_dis, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f075 = pd.DataFrame({"mean": [np.mean(avg_ncf_real)] + list(np.mean(avg_ncf_075, axis = 1)),
                                               "std": [np.std(avg_ncf_real)] + list(np.std(avg_ncf_075, axis = 1))},
                                              index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f100 = pd.DataFrame({"mean": [np.mean(avg_ncf_real)] + list(np.mean(avg_ncf_100, axis = 1)),
                                               "std": [np.std(avg_ncf_real)] + list(np.std(avg_ncf_100, axis = 1))},
                                              index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f125 = pd.DataFrame({"mean": [np.mean(avg_ncf_real)] + list(np.mean(avg_ncf_125, axis = 1)),
                                               "std": [np.std(avg_ncf_real)] + list(np.std(avg_ncf_125, axis = 1))},
                                              index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f150 = pd.DataFrame({"mean": [np.mean(avg_ncf_real)] + list(np.mean(avg_ncf_150, axis = 1)),
                                               "std": [np.std(avg_ncf_real)] + list(np.std(avg_ncf_150, axis = 1))},
                                              index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                    
        col = fairness_clf.index(f)
        ax = axes[c]

        # Plotting a dashed line for the individual fairness metrics.
        ax.axhline(0, 0, 5, ls = "--", c = "black")
    
        # Create line plots for the current metric: Adult, Bank, Credit & Average.
        ax.plot([0, 11], [results.iloc[0][0]]*len([0, 11]), color = colors[0])   # original
        ax.plot(epsi, results.iloc[0][1:5], color = colors[1], marker = markers[0], markersize = msize)   # DP-GAN
        # Dataset Fair DP-GANs.
        ax.plot(epsi, results_fdis.iloc[0][1:5], color = colors[2], marker = markers[1], markersize = msize)   
        ax.plot(epsi, results_f075.iloc[0][1:5], color = colors[3], marker = markers[2], markersize = msize)
        ax.plot(epsi, results_f100.iloc[0][1:5], color = colors[4], marker = markers[3], markersize = msize)
        ax.plot(epsi, results_f125.iloc[0][1:5], color = colors[5], marker = markers[4], markersize = msize)
        ax.plot(epsi, results_f150.iloc[0][1:5], color = colors[6], marker = markers[5], markersize = msize)
            
        # Customizing the plot.
        if c == 0:
            ax.set_ylabel(fairness_labels[col])
        ax.set_aspect('auto')
        ax.set_title(f"{dataset}" + r" ($\bf{T}$)")
        ax.set_xlabel(r"$\epsilon$")
        ax.set_xlim([0.6, 8.6])
        ax.set_ylim(limits[f][0][0], limits[f][0][1])
        ax.grid(True, linestyle = '--', alpha = 0.6)

        if f == "SF":
            h_patch = [mpatches.Patch(color = colors[i], label = labels[i]) for i in range(1)]
            h_lines = [mlines.Line2D([], [], color = colors[i], marker = markers[i - 1], label = labels[i], linestyle='None',
                        markersize = 10) for i in range(1, len(colors))]
            plt.figlegend(handles = h_patch + h_lines, ncol = 7, loc = "upper center", bbox_to_anchor = (0.5, 1.01))
            
        plt.tight_layout(rect = [0, 0, 1, 0.95])
    
    plt.show()

## Comparison of GANs' Results for Individual Fairness

In [ ]:
#adult_dis
#bank_dis
#credit_dis

#adult_075
#bank_075
#credit_075

#adult_100
#bank_100
#credit_100

#adult_125
#bank_125
#credit_125

#adult_150
#bank_150
credit_150

In [ ]:
#adult_DP
#bank_DP
credit_DP